In [37]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [48]:
CSV_PATH = '../taegram_all_table_merged_2018_2026.csv'

cols = [
    # Identifier
    'vin_hin_no',

    # Vehicle identity
    'make', 'model', 'year', 'trim', 'us_styles',
    'vehicle_category', 'body_type', 'body_subtype', 'color',

    # Powertrain
    'engines_name', 'ice_displacement', 'ice_cylinders', 'ice_block_type', 'ice_max_hp',
    'transmissions_name',

    # Usage
    'mileage',

    # Condition
    'vehicle_cond_picklist_id', 'vehicle_cond_picklist_id_name',
    'engine_cond_picklist_id', 'engine_cond_picklist_id_name',
    'transmission_cond_picklist_id', 'transmission_cond_picklist_id_name',
    'body_paint_cond_picklist_id', 'body_paint_cond_picklist_id_name',
    'interior_cond_picklist_id', 'interior_cond_picklist_id_name',
    'tire_cond_picklist_id', 'tire_cond_picklist_id_name',
    'other_damage_pklist_id', 'other_damage_pklist_id_name',

    # Title / legal
    'state_title_picklist', 'state_title_picklist_name',

    # Location & logistics
    'state_picklist_id', 'state_picklist_id_name', 'zip',
    'located_at_donation_c_a', 'accessible_for_tow_truck', 'speciality_item',

    # Target & metadata
    'sale_value', 'creation_datetime', 'comment',
]

# usecols requires every name to exist in the file's header — check first so a
# missing/renamed column doesn't blow up the read. This file is 3.7GB / 267
# cols (that's what OOM-killed the kernel last time), so we only ever want
# this narrow slice, not the full table.
header = pd.read_csv(CSV_PATH, nrows=0).columns
missing = [c for c in cols if c not in header]
cols_present = [c for c in cols if c in header]
if missing:
    print(f"Not found in CSV header, skipping: {missing}")

# usecols ignores the order you pass — pandas returns them in file order,
# so re-apply the logical order after the read.
df = pd.read_csv(CSV_PATH, usecols=cols_present)[cols_present]
df.shape

/tmp/ipykernel_12352/3398800920.py:50: DtypeWarning: Columns (11,17,23,33,34,40,95,116,218,237) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH, usecols=cols_present)[cols_present]


(931347, 42)

In [49]:
# Raw DataOne-sourced columns (see train_save_script21.py's DATAONE_FEATURES /
# --use-dataone gate, default off) -- drop them here so this EDA slice
# matches what the deployed script21 model actually trains on.
DATAONE_COLS = [
    'body_type', 'engines_name', 'ice_block_type', 'ice_cylinders',
    'ice_displacement', 'ice_max_hp', 'transmissions_name', 'us_styles',
]
df = df.drop(columns=[c for c in DATAONE_COLS if c in df.columns])
df.shape

(931347, 34)

In [50]:
df.head(3)

,vin_hin_no,make,model,year,trim,vehicle_category,body_subtype,color,mileage,vehicle_cond_picklist_id,vehicle_cond_picklist_id_name,engine_cond_picklist_id,engine_cond_picklist_id_name,transmission_cond_picklist_id,transmission_cond_picklist_id_name,body_paint_cond_picklist_id,body_paint_cond_picklist_id_name,interior_cond_picklist_id,interior_cond_picklist_id_name,tire_cond_picklist_id,tire_cond_picklist_id_name,other_damage_pklist_id,other_damage_pklist_id_name,state_title_picklist,state_title_picklist_name,state_picklist_id,state_picklist_id_name,zip,located_at_donation_c_a,accessible_for_tow_truck,speciality_item,sale_value,creation_datetime,comment
0,1GNFK16R9XJ536845,Chevrolet,Suburban,1999.0,NaN,NaN,NaN,22947.0,281816.0,22968.0,Runs & Drives,23059.0,Unknown,23066.0,Unknown,23045.0,Unknown,23052.0,Unknown,23067.0,Unknown,NaN,NaN,13347.0,North Carolina,13347.0,North Carolina,27527,True,True,NaN,NaN,2018-01-02 04:12:00,NaN
1,2T1AE09B7RC069281,Toyota,Corolla,1994.0,NaN,NaN,NaN,22947.0,140000.0,22968.0,Runs & Drives,23059.0,Unknown,23066.0,Unknown,23045.0,Unknown,23052.0,Unknown,23067.0,Unknown,NaN,NaN,13357.0,Virginia,13357.0,Virginia,22312,True,True,NaN,125.0,2018-01-02 04:24:00,NaN
2,1G1NE52M0X6267336,Chevrolet,Malibu,1999.0,NaN,NaN,NaN,22947.0,228773.0,22968.0,Runs & Drives,23059.0,Unknown,23066.0,Unknown,23045.0,Unknown,23052.0,Unknown,23067.0,Unknown,NaN,NaN,13327.0,Illinois,13327.0,Illinois,60156,False,True,NaN,250.0,2018-01-02 04:32:00,NaN


In [43]:
df.dtypes

year                                  float64
make                                   object
model                                  object
trim                                   object
color                                 float64
vin_hin_no                             object
mileage                               float64
accessible_for_tow_truck               object
state_title_picklist                  float64
located_at_donation_c_a                object
sale_value                            float64
body_subtype                           object
vehicle_category                       object
speciality_item                        object
state_picklist_id                     float64
zip                                    object
vehicle_cond_picklist_id              float64
body_paint_cond_picklist_id           float64
interior_cond_picklist_id             float64
engine_cond_picklist_id               float64
transmission_cond_picklist_id         float64
tire_cond_picklist_id             

In [44]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,907501.0,NaN,NaN,NaN,2001.688214,99.176441,0.0,1999.0,2003.0,2006.0,92618.0
make,906981,6020,Toyota,111009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,902762,16495,Accord,27783,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trim,129175,2122,Base,22137,NaN,NaN,NaN,NaN,NaN,NaN,NaN
color,900460.0,NaN,NaN,NaN,22935.473842,601.557822,0.0,22948.0,22951.0,22954.0,22959.0
vin_hin_no,875280,824682,N0V1N,246,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mileage,920325.0,NaN,NaN,NaN,774877.565044,426119936.954578,-1.0,87000.0,146518.0,195482.0,347823577570.0
accessible_for_tow_truck,929264,2,True,853003,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state_title_picklist,881824.0,NaN,NaN,NaN,13321.226085,435.917855,0.0,13318.0,13334.0,13349.0,24468.0
located_at_donation_c_a,931342,2,True,733007,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
for col in df.columns:
    print(f"=== {col} ({df[col].nunique()} unique, {df[col].notna().sum()} non-null) ===")
    print(df[col].value_counts())
    print()

=== vin_hin_no (824682 unique, 875280 non-null) ===
vin_hin_no
N0V1N                246
NOVIN                221
0                    169
UNKNOWN              120
NONE                 107
                    ... 
5TDZT34AX1S055573      1
1GNDT13WXY2213673      1
KL1TD66616B667664      1
EMD12016C999           1
1FMDU63K95UA85728      1
Name: count, Length: 824682, dtype: int64

=== make (6020 unique, 906981 non-null) ===
make
Toyota           111009
Ford              96834
Honda             96040
Chevrolet         63471
Nissan            40993
                  ...  
SPC                   1
BRAXTON CREEK         1
Jester                1
Kansas                1
reliance              1
Name: count, Length: 6020, dtype: int64

=== model (16495 unique, 902762 non-null) ===
model
Accord                27783
Camry                 27183
Civic                 25280
Corolla               18230
Prius                 16023
                      ...  
Swifty 12                 1
T3 Utility Traile

In [52]:
print(df.columns)

Index(['vin_hin_no', 'make', 'model', 'year', 'trim', 'vehicle_category',
       'body_subtype', 'color', 'mileage', 'vehicle_cond_picklist_id',
       'vehicle_cond_picklist_id_name', 'engine_cond_picklist_id',
       'engine_cond_picklist_id_name', 'transmission_cond_picklist_id',
       'transmission_cond_picklist_id_name', 'body_paint_cond_picklist_id',
       'body_paint_cond_picklist_id_name', 'interior_cond_picklist_id',
       'interior_cond_picklist_id_name', 'tire_cond_picklist_id',
       'tire_cond_picklist_id_name', 'other_damage_pklist_id',
       'other_damage_pklist_id_name', 'state_title_picklist',
       'state_title_picklist_name', 'state_picklist_id',
       'state_picklist_id_name', 'zip', 'located_at_donation_c_a',
       'accessible_for_tow_truck', 'speciality_item', 'sale_value',
       'creation_datetime', 'comment'],
      dtype='object')
